# Export map tiles for gain and loss — GMW v4

Exports one combined gain/loss tileset per year from the unified GMW v4
ImageCollection (`mangrove_change_ic_v4`). Each image has a single
`classification` band where gain=1 and loss=2. Both classes are rendered
at all zoom levels (0–12).

In [10]:
import ee
import geemap
%run utils.ipynb

In [11]:
ee.Authenticate(auth_mode="notebook")

True

In [12]:
ee.Initialize(project='mangrove-atlas-246414')

### Configuration

In [13]:
gcs_bucket = 'mangrove_atlas'
ic_asset   = 'projects/mangrove-atlas-246414/assets/land-cover/mangrove_change_ic_v4'

# All 40 change years available in the v4 IC (1986–2025).
# Reduce this list to export a subset.
data_year_range = list(range(1986, 2026))

region = ee.Geometry.BBox(-179, -40, 179, 40)

### Load collection

In [14]:
ic = ee.ImageCollection(ic_asset).sort('system:time_start')

print(f'IC size: {ic.size().getInfo()} images')
print(f'Change years: {ic.aggregate_array("change_year").sort().getInfo()}')

IC size: 40 images
Change years: [1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


### Style

In [15]:
gain_loss_ramp = '''
    <RasterSymbolizer>
        <ColorMap type="values" extended="false">
            <ColorMapEntry color="#a6cb10" quantity="1" label="Gain"/>
            <ColorMapEntry color="#eb6240" quantity="2" label="Loss"/>
        </ColorMap>
    </RasterSymbolizer>
'''

### Visualise prior to export

In [21]:
Map = geemap.Map(center=(-5, 39), zoom=5, basemap='CartoDB.PositronNoLabels')
#render the last year of the IC (2025) for visualization

last_img = ee.Image(ic.sort('system:time_start', False).first())

Map.addLayer(
    last_img.clip(region).sldStyle(gain_loss_ramp),
    {}, 'Gain + Loss (last year)', True, 1
)
Map.addLayer(region, {}, 'Export region', True, 0.3)

Map

Map(center=[-5, 39], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tran…

### Export map tiles

In [ ]:
export_tasks = exportMapTasks(
    ic, 'gain-loss-v4', gcs_bucket, data_year_range,
    region, gain_loss_ramp, 0, 12, env='staging', key='change_year'
)

In [ ]:
print(f'Total tasks to submit: {len(export_tasks)}')
batchExecute(export_tasks)

### Monitor tasks

In [ ]:
from collections import Counter

all_ops  = ee.data.listOperations()
relevant = [
    t for t in all_ops
    if t.get('metadata', {}).get('description', '').startswith('gain-loss-v4_')
]

status_counts = Counter(
    t.get('metadata', {}).get('state', 'UNKNOWN') for t in relevant
)
print('Task status summary:', dict(status_counts))

failed = [
    (t['metadata']['description'], t['metadata'].get('errorMessage', 'no message'))
    for t in relevant
    if t.get('metadata', {}).get('state') == 'FAILED'
]
if failed:
    print('\nFailed tasks:')
    for name, msg in failed:
        print(f'  {name}: {msg}')
else:
    print('No failed tasks.')